# India crop production and rainfall (subdivision-year)
Joins district crop production to IMD meteorological-subdivision rainfall using only the Python standard library so it runs offline with no packages.

1. Map `State_Name` + `District_Name` through `data/india-district-subdivision.json`.
2. Sum crop `Production` to `imd_subdivision` + year.
3. Sum monthly rainfall columns to a yearly total.
4. Inner-join on `imd_subdivision` + `year` (one-to-one).

Replace the sample paths with the full Kaggle crop file and the data.gov.in rainfall resource after download. Unmatched districts are dropped, not guessed. Result grain is **subdivision**, not district.


In [ ]:
import csv, json, re
from collections import defaultdict
from pathlib import Path

root = Path(".")
cross = json.loads((root / "data/india-district-subdivision.json").read_text())
norm = lambda s: re.sub(r"[^A-Z0-9]+", " ", str(s).upper()).strip()

def subdivision(state, district):
    key = f"{norm(state)}|{norm(district)}"
    return cross["byDistrict"].get(key) or cross["byState"].get(norm(state))

crop_rows = list(csv.DictReader((root / "data/samples/india-crop-sample.csv").open()))
matched = 0
crop_sum = defaultdict(float)
for row in crop_rows:
    sub = subdivision(row["State_Name"], row["District_Name"])
    if not sub:
        continue
    matched += 1
    crop_sum[(sub, int(row["Crop_Year"]))] += float(row["Production"])
rate = matched / len(crop_rows)
assert rate >= 0.8, f"crosswalk match rate {rate:.0%} below 80%"

months = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]
rain_sum = {}
for row in csv.DictReader((root / "data/samples/india-rainfall-sample.csv").open()):
    key = (row["SUBDIVISION"], int(row["YEAR"]))
    rain_sum[key] = sum(float(row[m] or 0) for m in months)

joined = []
for key, production in sorted(crop_sum.items()):
    if key not in rain_sum:
        continue
    joined.append((key[0], key[1], production, rain_sum[key]))
assert joined, "no overlapping subdivision-year keys"
print("imd_subdivision year crop_production rainfall_mm")
for row in joined:
    print(*row)
print(f"crosswalk match rate {rate:.0%}; {len(joined)} subdivision-year rows")
